# Scalar and tensor spectra

Calculate N2LO spectra (`p=2`) with the Python API. Install SRI with
`pip install -e .` and select that environment in your notebook editor.

Formulas are grouped by slow-roll weight. Displayed decimals use six significant
digits; `digits=None` shows all stored digits. Returned expressions and exports
keep their original precision.

In [1]:
from pathlib import Path
import sys

root = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "utils/notebook_display.py").is_file()), None)
if root is None:
    raise FileNotFoundError("Open this notebook inside the SRI repository.")
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
from utils.notebook_display import show_equation, show_spectra, show_consistency

import sympy as sp
from IPython.display import display
from sri import spectra
from sri import serialization

sp.init_printing()


## Options

- `p=2`: N2LO spectra.
- `method="iterative"` (default) or `"closed"`: background algebra.
- `symbolic=False` (default) or `True`: numerical or exact coefficients.
- `kinflation=False` (default) or `True`: canonical inflation or variable sound speed.
- `coefficient_tolerance=1e-10`: numerical coefficient convergence only.

In [2]:
p = 2
method = "iterative"
symbolic = False
kinflation = False
coefficient_tolerance = 1e-10

options = dict(method=method, symbolic=symbolic, kinflation=kinflation,
               coefficient_tolerance=coefficient_tolerance)


## Generate scalar and tensor spectra

`.scalar` and `.tensor` include the normalizations. The relative factors and
truncated logarithms are available as `scalar_correction`, `tensor_correction`,
`log_scalar_correction` and `log_tensor_correction`.

In [3]:
power = spectra(p=p, **options)
show_spectra(power)


<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

## Compare the algebra methods

Generate exact spectra with `symbolic=True` using both methods and check their equality.

In [4]:
exact_power = {m: spectra(p=p, method=m, symbolic=True, kinflation=kinflation)
               for m in ("iterative", "closed")}
residuals = {field: sp.expand(getattr(exact_power["iterative"], field)
                             - getattr(exact_power["closed"], field))
             for field in ("scalar_correction", "tensor_correction")}
assert all(value == 0 for value in residuals.values())
print(residuals)
show_equation(r"F_t", exact_power[method].tensor_correction)


{'scalar_correction': 0, 'tensor_correction': 0}


<IPython.core.display.Math object>

## Numerical evaluation

`epsilon=[epsilon_1,epsilon_2,...]`. With `Mpl=1`, `H` is in reduced Planck units.
The example supplies enough flow parameters for `p=2` or `3`.

In [5]:
epsilon = [0.005, 0.02, 0.01, 0.005]
sound = [0.002, 0.0001] + [0.0] * (p + 1)
sound_inputs = dict(cs=0.7, sound=sound) if kinflation else {}
print(power.evaluate(H=1e-5, epsilon=epsilon, Mpl=1, **sound_inputs))


{'scalar': 2.564080767204516e-10, 'tensor': 2.0209722190039956e-11}


## Variable sound speed

`kinflation=True` selects k-inflation. Supply positive `cs` and
`sound=[alpha_0^cs,alpha_1^cs,...]`: `alpha_0^cs=d ln(c_s)/d ln(k)`;
higher entries are its successive logarithmic scale derivatives.

In [6]:
power_cs = spectra(p=p, method=method, symbolic=symbolic, kinflation=True,
                   coefficient_tolerance=coefficient_tolerance)
print(power_cs.evaluate(H=1e-5, epsilon=epsilon, cs=0.7, sound=sound))
print("Model:", power_cs.model, "Method:", power_cs.method)


{'scalar': 3.695695313460577e-10, 'tensor': 2.0209722190039956e-11}
Model: kinflation Method: iterative


## LaTeX and YAML

Formatting uses the computed expressions. File-writing lines are optional
and commented out.

In [7]:
latex_scalar = sp.latex(power.scalar)
latex_tensor = sp.latex(power.tensor)
yaml_text = serialization.dumps({
    "model": power.model, "method": power.method, "p": p,
    "coefficients": power.coefficient_metadata,
    "scalar": str(power.scalar), "tensor": str(power.tensor),
})
# from pathlib import Path
# Path("spectra.yaml").write_text(yaml_text)
# Path("scalar.tex").write_text(latex_scalar)


## Corresponding CLI options

| Python | CLI |
| --- | --- |
| `p=2` | `-p 2` |
| `method="iterative"` (default) | No algebra flag (default) |
| `method="closed"` | `--closed` |
| `symbolic=False` (default) | No coefficient flag (default) |
| `symbolic=True` | `--symbolic` |
| `kinflation=True` | `--kinflation` |
| `coefficient_tolerance=1e-10` | `--coefficient-tolerance 1e-10` |
| LaTeX / YAML and explicit file writing | `--format latex` / `yaml`, `--output PATH` |

See the [consistency notebook](02_consistency_relations.ipynb) for relations and
a `simplify` example, or the [API guide](../docs/api.md) for conventions.